# Handling Missing Values Using Drop and Fill Strategies

This milestone focuses on **handling missing values** in Pandas using drop and fill strategies.

After identifying missing data, the next step is to decide how to handle it responsibly — either by **removing** incomplete data or **filling** missing values in a meaningful way.

**Dataset:** Air Quality readings across major Indian cities (PM2.5, PM10, NO2, CO, AQI, Temperature, Humidity)

---

### What You Will Learn
| Goal | Method |
|------|--------|
| Remove incomplete rows | `dropna()` |
| Remove sparse columns | `dropna(axis=1, thresh=...)` |
| Fill with a constant | `fillna(value)` |
| Fill with mean/median | `fillna(df['col'].mean())` |
| Fill with mode | `fillna(df['col'].mode()[0])` |
| Forward / backward fill | `ffill()`, `bfill()` |

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

## Step 2 — Load the Dataset

We load an air quality CSV that intentionally contains missing values across several columns.

In [ ]:
# ── Load the raw air quality data with missing values ──────────────────────
file_path = '../data/raw/air_quality_missing.csv'
df = pd.read_csv(file_path)

print(f"Shape: {df.shape}  →  {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head(10))

## Step 3 — Detect Missing Values

Before handling missing values, always **detect and measure** them first.

> Rule: Never clean data you haven't inspected.

In [ ]:
# ── 3a. Total missing values per column ───────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %'    : missing_pct
})

print("╔══ Missing Values per Column ══╗")
display(missing_summary[missing_summary['Missing Count'] > 0])

In [ ]:
# ── 3b. Overall picture ───────────────────────────────────────────────────
total_cells   = df.shape[0] * df.shape[1]
total_missing = df.isnull().sum().sum()

print(f"Total cells   : {total_cells}")
print(f"Missing cells : {total_missing}")
print(f"Overall missing rate: {total_missing / total_cells * 100:.2f}%")
print()
print("Rows with at least one missing value:", df.isnull().any(axis=1).sum())
print("Rows that are completely clean       :", df.notnull().all(axis=1).sum())

In [ ]:
# ── 3c. Check dtypes — important before choosing fill strategy ────────────
print("Column dtypes:")
print(df.dtypes)
print()
print("df.info() summary:")
df.info()

---
## Step 4 — Dropping Missing Values

**When to drop?**
- The missing column is critical and cannot be estimated (e.g., the target variable)
- Very few rows are affected — removing them won't distort the dataset
- A column has so many missing values it carries no usable information

> Warning: Dropping reduces your data. Always compare shape before and after.

In [ ]:
# ── 4a. Drop ANY row that has at least one missing value ──────────────────
df_drop_any = df.dropna()

print("Shape BEFORE drop (any):", df.shape)
print("Shape AFTER  drop (any):", df_drop_any.shape)
print(f"Rows removed: {df.shape[0] - df_drop_any.shape[0]}")
print(f"Data retained: {df_drop_any.shape[0] / df.shape[0] * 100:.1f}%")
display(df_drop_any.head())

In [ ]:
# ── 4b. Drop rows missing in SPECIFIC critical columns ───────────────────
# AQI is the key air quality metric — rows without it are not useful
df_drop_aqi = df.dropna(subset=['aqi'])

print("Shape BEFORE (drop subset=['aqi']):", df.shape)
print("Shape AFTER  (drop subset=['aqi']):", df_drop_aqi.shape)
print(f"Rows removed: {df.shape[0] - df_drop_aqi.shape[0]}")

In [ ]:
# ── 4c. Drop rows where ALL values are missing (entirely empty rows) ──────
df_drop_all = df.dropna(how='all')

print("Rows where ALL values are NaN:", df.shape[0] - df_drop_all.shape[0])
print("Shape after removing all-NaN rows:", df_drop_all.shape)

In [ ]:
# ── 4d. Drop COLUMNS with more than 20% missing data ─────────────────────
threshold = int(0.80 * len(df))   # at least 80% of values must be present
df_drop_cols = df.dropna(axis=1, thresh=threshold)

print("Columns BEFORE:", df.shape[1], "→", list(df.columns))
print("Columns AFTER :", df_drop_cols.shape[1], "→", list(df_drop_cols.columns))
print("Columns removed:", set(df.columns) - set(df_drop_cols.columns))

In [ ]:
# ── 4e. Impact Summary ────────────────────────────────────────────────────
impact = pd.DataFrame({
    'Strategy'      : ['Original', 'Drop any row', 'Drop on AQI', 'Drop all-NaN rows', 'Drop sparse cols'],
    'Rows'          : [df.shape[0], df_drop_any.shape[0], df_drop_aqi.shape[0],
                       df_drop_all.shape[0], df_drop_cols.shape[0]],
    'Cols'          : [df.shape[1], df_drop_any.shape[1], df_drop_aqi.shape[1],
                       df_drop_all.shape[1], df_drop_cols.shape[1]],
    'Data Kept %'   : [
        '100%',
        f"{df_drop_any.shape[0]/df.shape[0]*100:.1f}%",
        f"{df_drop_aqi.shape[0]/df.shape[0]*100:.1f}%",
        f"{df_drop_all.shape[0]/df.shape[0]*100:.1f}%",
        f"{df_drop_cols.shape[0]/df.shape[0]*100:.1f}%",
    ]
})

print("Drop Strategy Impact Summary:")
display(impact)

---
## Step 5 — Filling Missing Values

**When to fill?**
- You cannot afford to lose rows (small dataset)
- The column has a sensible substitute value
- Missing percentage is moderate and the column is important

| Scenario | Fill method |
|----------|-------------|
| Numeric, roughly symmetric | Mean |
| Numeric, skewed / outliers | Median |
| Categorical / text | Mode |
| Known default / sentinel | Constant |
| Time-ordered data | Forward/backward fill |

In [ ]:
# ── 5a. Fill with a CONSTANT ──────────────────────────────────────────────
# When no sensor reading exists, 0 is a safe sentinel for 'co' (low level)
df_fill_const = df.copy()
df_fill_const['co'] = df_fill_const['co'].fillna(0)

print("Missing 'co' values BEFORE:", df['co'].isnull().sum())
print("Missing 'co' values AFTER (fill 0):", df_fill_const['co'].isnull().sum())
print()
print("'co' column stats after filling:")
print(df_fill_const['co'].describe())

In [ ]:
# ── 5b. Fill with MEAN ────────────────────────────────────────────────────
# PM2.5 is roughly normally distributed → mean is appropriate
df_fill_mean = df.copy()
pm25_mean = df_fill_mean['pm25'].mean().round(2)
df_fill_mean['pm25'] = df_fill_mean['pm25'].fillna(pm25_mean)

print(f"PM2.5 Mean used for fill: {pm25_mean}")
print("Missing 'pm25' BEFORE:", df['pm25'].isnull().sum())
print("Missing 'pm25' AFTER :", df_fill_mean['pm25'].isnull().sum())
print()
print("'pm25' stats before vs after:")
comparison = pd.DataFrame({
    'Before': df['pm25'].describe(),
    'After (mean fill)': df_fill_mean['pm25'].describe()
})
display(comparison)

In [ ]:
# ── 5c. Fill with MEDIAN ──────────────────────────────────────────────────
# AQI can have high-end outliers (pollution spikes) → median is safer
df_fill_median = df.copy()
aqi_median = df_fill_median['aqi'].median()
df_fill_median['aqi'] = df_fill_median['aqi'].fillna(aqi_median)

print(f"AQI Median used for fill: {aqi_median}")
print("Missing 'aqi' BEFORE:", df['aqi'].isnull().sum())
print("Missing 'aqi' AFTER :", df_fill_median['aqi'].isnull().sum())
print()
print("'aqi' stats after median fill:")
print(df_fill_median['aqi'].describe())

In [ ]:
# ── 5d. Fill with MODE (categorical data) ────────────────────────────────
# 'station' is a categorical column — mode is the only sensible choice
df_fill_mode = df.copy()
station_mode = df_fill_mode['station'].mode()[0]
df_fill_mode['station'] = df_fill_mode['station'].fillna(station_mode)

print(f"Station Mode used for fill: '{station_mode}'")
print("Missing 'station' BEFORE:", df['station'].isnull().sum())
print("Missing 'station' AFTER :", df_fill_mode['station'].isnull().sum())

In [ ]:
# ── 5e. Forward Fill (ffill) ─────────────────────────────────────────────
# Time-series data: carry the last valid reading forward
df_ffill = df.copy()

# Sort by city + date to make time order meaningful
df_ffill = df_ffill.sort_values(['city', 'date']).reset_index(drop=True)

print("Missing values BEFORE ffill:")
print(df_ffill[['pm25', 'pm10', 'no2', 'co', 'aqi']].isnull().sum())

df_ffill[['pm25', 'pm10', 'no2', 'co', 'aqi']] = (
    df_ffill[['pm25', 'pm10', 'no2', 'co', 'aqi']]
    .ffill()
)

print("\nMissing values AFTER ffill:")
print(df_ffill[['pm25', 'pm10', 'no2', 'co', 'aqi']].isnull().sum())

In [ ]:
# ── 5f. Backward Fill (bfill) ────────────────────────────────────────────
# Fills missing with the NEXT valid reading — useful at the start of series
df_bfill = df.sort_values(['city', 'date']).copy().reset_index(drop=True)
df_bfill[['pm25', 'pm10', 'no2', 'co', 'aqi']] = (
    df_bfill[['pm25', 'pm10', 'no2', 'co', 'aqi']]
    .bfill()
)

print("Missing values AFTER bfill:")
print(df_bfill[['pm25', 'pm10', 'no2', 'co', 'aqi']].isnull().sum())

## Step 6 — Applying a Combined Fill Strategy (All Columns Clean)

In practice, you apply **different strategies to different columns** based on their type and importance.

In [ ]:
# ── Full cleaning pipeline on a fresh copy ───────────────────────────────
df_clean = df.copy()

# Numeric columns: fill with median (robust to outliers)
numeric_cols = ['pm25', 'pm10', 'no2', 'co', 'aqi', 'temperature', 'humidity']
for col in numeric_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"  Filled '{col}' with median = {median_val:.2f}")

# Categorical column: fill with mode
station_mode = df_clean['station'].mode()[0]
df_clean['station'] = df_clean['station'].fillna(station_mode)
print(f"  Filled 'station' with mode = '{station_mode}'")

print("\n─── Final missing value check ───")
remaining = df_clean.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "No missing values remain ✓")

print("\nClean DataFrame shape:", df_clean.shape)
display(df_clean.head())

---
## Step 7 — Drop vs Fill: Side-by-Side Comparison

Let's directly compare the two main strategies for the same dataset.

In [ ]:
# ── Approach A: Drop all rows with any missing value ──────────────────────
df_approach_drop = df.dropna().copy()

# ── Approach B: Fill all numeric cols with median, categorical with mode ──
df_approach_fill = df.copy()
for col in numeric_cols:
    df_approach_fill[col] = df_approach_fill[col].fillna(df_approach_fill[col].median())
df_approach_fill['station'] = df_approach_fill['station'].fillna(df_approach_fill['station'].mode()[0])

# ── Summary comparison ────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Metric'          : ['Total Rows', 'Missing Values', 'PM2.5 Mean', 'PM2.5 Median', 'AQI Mean', 'AQI Median'],
    'Original'        : [
        df.shape[0], df.isnull().sum().sum(),
        round(df['pm25'].mean(), 2), round(df['pm25'].median(), 2),
        round(df['aqi'].mean(), 2), round(df['aqi'].median(), 2)
    ],
    'After Drop'      : [
        df_approach_drop.shape[0], 0,
        round(df_approach_drop['pm25'].mean(), 2), round(df_approach_drop['pm25'].median(), 2),
        round(df_approach_drop['aqi'].mean(), 2), round(df_approach_drop['aqi'].median(), 2)
    ],
    'After Fill (Med)': [
        df_approach_fill.shape[0], 0,
        round(df_approach_fill['pm25'].mean(), 2), round(df_approach_fill['pm25'].median(), 2),
        round(df_approach_fill['aqi'].mean(), 2), round(df_approach_fill['aqi'].median(), 2)
    ]
})

print("Drop vs Fill — Side-by-Side Comparison")
display(comparison)

In [ ]:
# ── Insight: What did we lose vs what did we assume? ─────────────────────
rows_lost = df.shape[0] - df_approach_drop.shape[0]
rows_kept = df_approach_fill.shape[0]

print("=" * 50)
print("Strategy        | Rows Kept | Rows Lost | Missing Left")
print("-" * 50)
print(f"Drop (any NaN)  | {df_approach_drop.shape[0]:9} | {rows_lost:9} | 0")
print(f"Fill (median)   | {rows_kept:9} | {0:9} | 0")
print("=" * 50)
print()
print(f"Drop lost {rows_lost / df.shape[0] * 100:.1f}% of the dataset.")
print(f"Fill preserved all {rows_kept} rows but introduced assumed values.")

---
## Step 8 — Common Mistakes to Avoid

These are real errors beginners make. Each cell shows the wrong way and the right way.

In [ ]:
# ── Mistake 1: Filling categorical data with numeric values ───────────────
df_mistake = df.copy()

# WRONG — filling 'station' (text) with 0 makes no sense
wrong_fill = df_mistake['station'].fillna(0)
print("WRONG fill on 'station' (sample):", wrong_fill[wrong_fill == '0'].count(), "cells set to '0'")
print("Values after wrong fill:", wrong_fill.unique()[:5])

# RIGHT — use mode for categorical
correct_fill = df_mistake['station'].fillna(df_mistake['station'].mode()[0])
print("\nCORRECT fill on 'station' using mode:", df_mistake['station'].mode()[0])
print("Missing after correct fill:", correct_fill.isnull().sum())

In [ ]:
# ── Mistake 2: Dropping without checking impact ───────────────────────────
df_mistake2 = df.copy()

# How much data do we lose for each column when we drop that column's NaN?
print("Loss per column if we drop its missing rows:")
print()
for col in df_mistake2.columns:
    n_missing = df_mistake2[col].isnull().sum()
    if n_missing > 0:
        pct = n_missing / len(df_mistake2) * 100
        verdict = "⚠ High loss" if pct > 15 else "✓ Acceptable"
        print(f"  {col:12s}: {n_missing:2d} rows ({pct:5.1f}%)  {verdict}")

In [ ]:
# ── Mistake 3: Not checking results after cleaning ────────────────────────
df_cleaned_check = df_clean.copy()

# Always verify — don't assume fillna() worked on all columns
total_missing_after = df_cleaned_check.isnull().sum().sum()

if total_missing_after == 0:
    print("VERIFIED: No missing values remain in df_clean ✓")
else:
    print(f"WARNING: {total_missing_after} missing values still present!")
    print(df_cleaned_check.isnull().sum()[df_cleaned_check.isnull().sum() > 0])

In [ ]:
# ── Mistake 4: Using mean on skewed data (outlier distortion demo) ─────────
# AQI in Delhi spikes — mean gets pulled higher than the median
delhi = df[df['city'] == 'Delhi']['aqi']

mean_val   = delhi.mean()
median_val = delhi.median()

print(f"Delhi AQI — Mean: {mean_val:.1f}  |  Median: {median_val:.1f}")

if abs(mean_val - median_val) > 5:
    print("→ Mean and median differ significantly.")
    print("→ Use MEDIAN for AQI filling to avoid bias from spikes.")
else:
    print("→ Mean and median are close. Either is acceptable.")

---
## Step 9 — Decision Guide: Drop or Fill?

Use this framework to decide:

| Condition | Recommended Strategy |
|-----------|---------------------|
| Column missing > 50% of values | Drop the column |
| Row missing all values | Drop the row |
| < 5% rows affected, column critical | Drop the rows |
| Numeric, low skew | Fill with mean |
| Numeric, high skew / outliers | Fill with median |
| Categorical | Fill with mode |
| Time-ordered values | Forward or backward fill |
| Known default (zero reading) | Fill with constant |

In [ ]:

# ── Auto-generate decision recommendation per column ─────────────────────
print("Decision Guide for this dataset:")
print("-" * 60)

for col in df.columns:
    pct_missing = df[col].isnull().mean() * 100
    is_numeric  = pd.api.types.is_numeric_dtype(df[col])

    if pct_missing == 0:
        recommendation = "No action needed"
    elif pct_missing > 50:
        recommendation = "DROP the column (too sparse)"
    elif not is_numeric:
        recommendation = f"FILL with mode = '{df[col].mode()[0]}'"
    else:
        skew = abs(df[col].skew())
        if skew > 1.0:
            recommendation = f"FILL with median = {df[col].median():.2f} (skewed, skew={skew:.2f})"
        else:
            recommendation = f"FILL with mean = {df[col].mean():.2f} (symmetric, skew={skew:.2f})"

    print(f"  {col:12s} | {pct_missing:5.1f}% missing | {recommendation}")


---
## Step 10 — Save the Cleaned Dataset

In [ ]:
# ── Save df_clean (fill strategy) to the processed folder ────────────────
output_path = '../data/processed/air_quality_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Shape: {df_clean.shape}")

# Confirm zero missing
saved_df = pd.read_csv(output_path)
print(f"Missing values in saved file: {saved_df.isnull().sum().sum()}")
display(saved_df.head())

---
## Summary

### What Was Done

| Step | Action | Method |
|------|--------|--------|
| 1 | Loaded dataset | `pd.read_csv()` |
| 2 | Detected missing values | `isnull().sum()` |
| 3 | Dropped any-NaN rows | `dropna()` |
| 4 | Dropped on critical column | `dropna(subset=['aqi'])` |
| 5 | Dropped sparse columns | `dropna(axis=1, thresh=...)` |
| 6 | Filled with constant | `fillna(0)` |
| 7 | Filled with mean | `fillna(col.mean())` |
| 8 | Filled with median | `fillna(col.median())` |
| 9 | Filled with mode | `fillna(col.mode()[0])` |
| 10 | Forward/backward fill | `ffill()` / `bfill()` |
| 11 | Combined pipeline | All strategies together |
| 12 | Drop vs Fill comparison | Shape + stats comparison |
| 13 | Common mistakes | Wrong fill type, blind drop |
| 14 | Saved clean data | `to_csv()` |

### Key Takeaways
- **Always detect before you handle** — never guess at how much is missing
- **Dropping** is simple but costs you data — check percentage impact first
- **Mean** is best when data is symmetric; **Median** is safer when skewed
- **Mode** is the only correct choice for categorical columns
- **ffill/bfill** are powerful for time-series sensor data
- Always **verify** after cleaning — confirm zero missing values remain
- There is no single right answer — **justify your choices** based on context